<a href="https://colab.research.google.com/github/fenildhanani/Deep-Learning/blob/main/IMBD_Rating_with_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
import pandas as pd

df = pd.read_csv(
    "IMDB Dataset.csv",
    engine="python",
    on_bad_lines="skip"
)

print(df.head())
print(df.shape)

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(35631, 2)


In [55]:
df.head()

,review,sentiment
0,None,1
1,None,1
2,None,1
3,None,0
4,None,1


In [56]:
df["review"] = df["review"].str.lower()

Removing the URLs

In [45]:
import re

def remove_url(text):
  re.sub(r"http\S+"," ", text)

df["review"] = df["review"].apply(remove_url)

# 3.Removing puncuations

In [46]:
import re

df["review"] = df["review"].fillna("")

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)
    return text

df["review"] = df["review"].apply(remove_punctuations)

In [47]:
def remove_html(text):
    text = re.sub(r"[.*?]", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

# 5 removing the stepwords

In [48]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [49]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [58]:
def remove_stopwords(text):
  if not isinstance(text, str):
    return ""
  tokens = word_tokenize(text)
  stop_words = set(stopwords.words("english"))
  filtered_tokens = [word for word in tokens if word.lower() not in stop_words]
  return " ".join(filtered_tokens)


df["review"] = df["review"].apply(remove_stopwords)

# def remove_stopwords(text):
#   tokens = word_tokenize(text)
#   stop_words = stopwords.words("english")

#   for word in tokens:
#     for word in stop_words:
#        text = text.replace(word, "")
# df["review"] = df["review"].apply(remove_stopwords)


In [51]:
df.head()

,review,sentiment
0,None,positive
1,None,positive
2,None,positive
3,None,negative
4,None,positive


6.

In [65]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()


# stemming


In [36]:
from nltk.stem import PorterStemmer

In [60]:
def stemming(text):
  if not isinstance(text, str):
    return ""  # Return an empty string if the value is None/NaN

  ps = PorterStemmer()
  stemmed_words = []
  tokens = word_tokenize(text)

  for token in tokens:
    stemmed_token = ps.stem(token)
    stemmed_words.append(stemmed_token)

  return " ".join(stemmed_words)


df["review"] = df["review"].apply(stemming)

In [61]:
df.head()

,review,sentiment
0,one review mention watch 1 oz episod 'll hook ...,positive
1,wonder littl product . < br / > < br / > film ...,positive
2,thought wonder way spend time hot summer weeke...,positive
3,basic 's famili littl boy ( jake ) think 's zo...,negative
4,petter mattei 's `` love time money `` visual ...,positive


In [62]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [63]:
y =df["sentiment"]

In [64]:
y

,sentiment
0,1
1,1
2,1
3,0
4,1
...,...
35626,0
35627,0
35628,0
35629,1


In [59]:
# vectorizatiom

In [66]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [67]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3013636 stored elements and shape (35631, 5000)>

In [68]:
# Dataset $ dataloader


In [69]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [70]:
X_train.shape

(28504, 5000)

In [71]:
X_test.shape

(7127, 5000)

In [77]:
import torch
from torch.utils.data import TensorDataset, DataLoader


In [79]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [81]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [82]:
train_loader = DataLoader(train_set, batch_size = 64, shuffle = True)
train_loader = DataLoader(test_set, batch_size = 64, shuffle = True)

In [83]:
# build RNN

In [86]:
import torch.nn as nn
import torch.optim as optim

In [91]:
class RNN(nn.Module):
  def __init__(self, input_size, hidden_size=128, num_layers=1):
    super().__init__()

    self.hidden_size = hidden_size
    self.num_layers = num_layers

    # RNN layers
    self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
    # fully connected layer
    self.fc = nn.Linear(hidden_size, 1)

    # optional
  def forward(self, x):
    h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

    out,_ = self.rnn(x, h0)

    out = self.fc(out[:, -1, :])
    return out


In [99]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [110]:
epochs = 10

for epoch in range(epochs):
  model.train()

  for Xb, yb in train_loader:
    optimizer.zero_grad()

    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    outputs = torch.sigmoid(outputs.squeeze())

    loss = criterion(outputs, yb)
    loss.backward()
    optimizer.step()

print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")

epoch = 10/10 and loss = 0.0007405778160318732


In [113]:
# evaluat

model.eval()
with torch.no_grad():
  correct_vals = 0
  tot_vals = 0

  for Xb, yb in train_loader:
    Xb = Xb.unsqueeze(1)

    outputs = model(Xb)
    predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

    tot_vals += yb.size(0)
    correct_vals = (predicted == yb).sum().item()

  print(f"(accuracy ={correct_vals/tot_vals*100})")



(accuracy =0.32271643047565596)
